# Hugging Face -mallien testaus

Tässä notebookissa testataan kahta erilaista Hugging Face -mallia omalla testidatalla:

1. **`unitary/toxic-bert`**: tekstin toksisuuden luokittelu
2. **`distilbert/distilbert-base-uncased-finetuned-sst-2-english`**: tekstin sentimenttianalyysi

Mallit ajetaan valmiilla painoilla Hugging Facen `transformers`-kirjaston avulla. Testidata on itse määriteltyjä lyhyitä lauseita.


In [1]:
import time
import torch
from transformers import pipeline

start_time = time.time()

DEVICE = 0 if torch.cuda.is_available() else -1

print("PyTorch version:", torch.__version__)
print("Device:", "GPU" if DEVICE == 0 else "CPU")


PyTorch version: 2.9.1+cu129
Device: GPU


# Malli 1: toksisuuden tunnistus

Ensimmäinen malli on `unitary/toxic-bert`. Se on tekstiluokittelumalli, joka antaa pisteitä useille toksisuuden alaluokille, esimerkiksi `toxic`, `insult`, `threat` ja `identity_hate`.

Testaan mallia itse kirjoitetuilla lyhyillä lauseilla, joista osa on ystävällisiä ja osa selvästi negatiivisia.


In [2]:
toxicity_model_name = "unitary/toxic-bert"

toxicity_classifier = pipeline(
    task="text-classification",
    model=toxicity_model_name,
    tokenizer=toxicity_model_name,
    top_k=None,
    function_to_apply="sigmoid",
    device=DEVICE,
)

toxicity_texts = [
    "I hope you have a wonderful day!",
    "I hate you and you are awful.",
    "That was a bad decision, but we can fix it.",
    "You are stupid and nobody likes you.",
    "Thank you for helping me with the project.",
]

start = time.time()
toxicity_results = toxicity_classifier(toxicity_texts)
toxicity_runtime = time.time() - start

print(f"Toxicity model runtime: {toxicity_runtime:.2f} seconds")


Device set to use cuda:0


Toxicity model runtime: 0.58 seconds


In [3]:
print("Toxicity analysis results\n")

for text, result in zip(toxicity_texts, toxicity_results):
    sorted_result = sorted(result, key=lambda x: x["score"], reverse=True)
    highest = sorted_result[0]

    print("Text:", text)
    print("Highest toxicity label:", highest["label"])
    print("Highest score:", round(highest["score"], 4))
    print("All labels:")
    for item in sorted_result:
        print(f"  {item['label']:15s} {item['score']:.4f}")
    print("-" * 60)


Toxicity analysis results

Text: I hope you have a wonderful day!
Highest toxicity label: toxic
Highest score: 0.0007
All labels:
  toxic           0.0007
  insult          0.0002
  obscene         0.0002
  identity_hate   0.0002
  threat          0.0002
  severe_toxic    0.0001
------------------------------------------------------------
Text: I hate you and you are awful.
Highest toxicity label: toxic
Highest score: 0.985
All labels:
  toxic           0.9850
  insult          0.7836
  obscene         0.1697
  identity_hate   0.0131
  severe_toxic    0.0102
  threat          0.0033
------------------------------------------------------------
Text: That was a bad decision, but we can fix it.
Highest toxicity label: toxic
Highest score: 0.0007
All labels:
  toxic           0.0007
  insult          0.0002
  obscene         0.0002
  identity_hate   0.0001
  severe_toxic    0.0001
  threat          0.0001
------------------------------------------------------------
Text: You are stupid and

## Pohdinta mallista 1

Toxic-BERT reagoi selvästi voimakkaammin lauseisiin, joissa käytetään loukkaavia sanoja kuten *hate*, *awful* tai *stupid*. Ystävälliset lauseet saivat matalammat toksisuuspisteet.

Mallin tulokset ovat järkeviä lyhyillä englanninkielisillä testilauseilla, mutta mallia ei kannata tulkita täydellisenä totuutena. Esimerkiksi konteksti, ironia ja kulttuuriset ilmaukset voivat vaikuttaa siihen, miten toksiseksi teksti oikeasti pitäisi arvioida. Lisäksi malli on englanninkieliseen dataan painottunut, joten suomenkieliset lauseet eivät välttämättä toimisi yhtä luotettavasti.


# Malli 2: sentimenttianalyysi

Toinen malli on `distilbert/distilbert-base-uncased-finetuned-sst-2-english`. Se luokittelee tekstin sentimentin positiiviseksi tai negatiiviseksi.

Testaan mallia itse kirjoitetuilla englanninkielisillä lauseilla, joissa on positiivisia, negatiivisia ja hieman neutraalimpia ilmaisuja.


In [4]:
sentiment_model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

sentiment_classifier = pipeline(
    task="sentiment-analysis",
    model=sentiment_model_name,
    tokenizer=sentiment_model_name,
    device=DEVICE,
)

sentiment_texts = [
    "This course project was difficult, but I learned a lot.",
    "The model worked surprisingly well and the results were useful.",
    "The instructions were confusing and the training failed many times.",
    "I am happy with the final notebook.",
    "The result is not perfect, but it is good enough for this exercise.",
]

start = time.time()
sentiment_results = sentiment_classifier(sentiment_texts)
sentiment_runtime = time.time() - start

print(f"Sentiment model runtime: {sentiment_runtime:.2f} seconds")


Device set to use cuda:0


Sentiment model runtime: 0.06 seconds


In [5]:
print("Sentiment analysis results\n")

for text, result in zip(sentiment_texts, sentiment_results):
    print("Text:", text)
    print("Prediction:", result["label"])
    print("Score:", round(result["score"], 4))
    print("-" * 60)


Sentiment analysis results

Text: This course project was difficult, but I learned a lot.
Prediction: POSITIVE
Score: 0.9943
------------------------------------------------------------
Text: The model worked surprisingly well and the results were useful.
Prediction: POSITIVE
Score: 0.9997
------------------------------------------------------------
Text: The instructions were confusing and the training failed many times.
Prediction: NEGATIVE
Score: 0.9998
------------------------------------------------------------
Text: I am happy with the final notebook.
Prediction: POSITIVE
Score: 0.9999
------------------------------------------------------------
Text: The result is not perfect, but it is good enough for this exercise.
Prediction: POSITIVE
Score: 0.9998
------------------------------------------------------------


In [6]:
# Ajastimen loppu

full_time = time.time() - start_time
print(f"Notebookin ajamiseen kului {full_time}s")

Notebookin ajamiseen kului 2.613917350769043s


## Pohdinta mallista 2

Sentimenttimalli tunnisti selvästi positiiviset lauseet positiivisiksi ja negatiiviset negatiivisiksi. Esimerkiksi onnistumista ja tyytyväisyyttä kuvaavat lauseet saavat positiivisen luokan, kun taas epäonnistumista ja sekavuutta kuvaavat lauseet painottuvat negatiivisiksi.

Rajoituksena on, että malli antaa vain kaksi luokkaa: `POSITIVE` tai `NEGATIVE`. Sen vuoksi neutraalit tai kaksijakoiset lauseet voivat mennä jompaan kumpaan luokkaan, vaikka ihminen tulkitsisi ne tasapainoisemmiksi. Esimerkiksi lause, jossa sanotaan tehtävän olleen vaikea mutta opettavainen, sisältää sekä negatiivista että positiivista sävyä.


# Yhteenveto

Notebookissa ladattiin ja ajettiin kaksi eri Hugging Face -mallia:

- `unitary/toxic-bert` toksisuuden tunnistamiseen
- `distilbert/distilbert-base-uncased-finetuned-sst-2-english` sentimenttianalyysiin

Molempia malleja testattiin omilla tekstiesimerkeillä, tulokset tulostettiin näkyviin ja tuloksia pohdittiin erillisissä markdown-osioissa.
